# EEEM068 - Sagittal ACL ViT-Small from Scratch - Part 3

**Scope:** Model definition, loss, optimiser, scheduler, and key training settings

**Source notebook:** `sagittal_acl_scratch.ipynb`

**Notes**
- This notebook is a split component of the original final notebook.
- Some setup cells are intentionally repeated so each part is easier to understand in isolation.
- If you want to run the full pipeline end-to-end, use the original notebook or follow the README in sequence.

# EEEM068 — Knee MRI ACL Classification: ViT-Small from Scratch

## Key Constraints
- **No pretrained weights** — full training from scratch on MRNet only
- **Sagittal plane only** — binary ACL classification (positive / negative)
- **~21M parameters** — ViT-Small/16 architecture (embed_dim=384, 12 blocks, 6 heads)
- All regularisation is tuned for a small medical dataset (≈1130 training scans)

## Why training from scratch is hard here
ViT-Small normally needs ImageNet-scale data. With only 1130 scans, we compensate with:
1. **Strong augmentation** — aggressive random affine, H-flip, colour jitter, Gaussian blur
2. **MixUp + CutMix** — both applied stochastically for better generalisation
3. **Stochastic depth (DropPath)** — 0.15 rate, prevents transformer co-adaptation
4. **Heavy head dropout** — 0.5 / 0.25 in the MLP head
5. **Label smoothing** — 0.1 for binary BCE
6. **Focal loss** — focuses learning on hard/rare positive ACL cases (γ=2, α=0.75)
7. **Warmup + cosine decay** — 10-epoch warmup, essential for ViT from scratch
8. **Weight decay 0.05** — strong L2, following MAE/DINO from-scratch recipes
9. **WeightedRandomSampler** — ACL is only 18% positive, needs oversampling
10. **Gradient clipping** — max norm 1.0

## Parameter count breakdown
| Component | Parameters |
|---|---|
| Patch embedding (16×16 → 384) | 295,296 |
| Positional embedding (197 × 384) | 75,648 |
| CLS token | 384 |
| 12 × Transformer blocks | 21,233,664 |
| Layer norm | 768 |
| Classification head | 50,689 |
| **Total** | **~21.66M** |


## Section 1 — Install & Imports

In [48]:
!pip install -q timm kagglehub

In [49]:
import os, gc, math, random, warnings
from pathlib import Path
from collections import deque
from io import BytesIO
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

import timm

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, classification_report,
    balanced_accuracy_score
)

warnings.filterwarnings("ignore")

# Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.11.0+cu130
timm     : 1.0.27
CUDA     : True
GPU      : NVIDIA RTX 4000 Ada Generation


## Section 2 — Configuration

All hyperparameters in one place. Key differences from a pretrained setup:

- `WARMUP_EPOCHS = 10` — ViT from scratch needs a long warmup to stabilise attention weights
- `EPOCHS = 120` — needs many more epochs without ImageNet initialisation
- `WEIGHT_DECAY = 0.05` — strong L2, follows MAE/DINO from-scratch training recipes
- `DROP_PATH_RATE = 0.15` — slightly higher stochastic depth since no pretrained regularisation
- `BASE_LR = 1e-3` — from-scratch ViT uses higher base LR than fine-tuning
- `MIN_LR = 1e-6` — cosine decay floor
- **No progressive unfreezing** — entire model trains from epoch 1

In [50]:
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────
    OUT_DIR          = Path("./mrnet_scratch_acl")
    CHECKPOINT_PATH  = str(OUT_DIR / "best_vit_scratch_acl.pth")
    TENSORBOARD_DIR  = str(OUT_DIR / "tb_scratch_acl")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # ── Task ───────────────────────────────────────────────────────────────
    # Binary: ACL tear (positive=1) vs no ACL tear (negative=0)
    # Sagittal plane only
    PLANE       = "sagittal"
    LABEL       = "acl"
    NUM_CLASSES = 1  # binary sigmoid output

    # ── Model ──────────────────────────────────────────────────────────────
    IMAGE_SIZE     = 224
    MODEL_NAME     = "vit_small_patch16_224"  # 21.66M params
    DROP_PATH_RATE = 0.15  # higher than fine-tune (0.10) for scratch

    # ── MRI slice selection ────────────────────────────────────────────────
    TOP_K_SLICES = 8   # select from top-8 high-variance slices
    N_CHANNELS   = 3   # stack 3 of them as R,G,B

    # ── Training ───────────────────────────────────────────────────────────
    BATCH_SIZE  = 8     # reduced for GPU memory; use GRAD_ACCUM to keep effective batch=16
    GRAD_ACCUM  = 2     # gradient accumulation steps → effective batch = BATCH_SIZE * GRAD_ACCUM
    EPOCHS      = 120
    MIN_EPOCHS  = 40
    PATIENCE    = 20
    NUM_WORKERS = 0     # 0 = main process only — avoids pin_memory OOM on shared/limited GPUs
    PIN_MEMORY  = False # pin_memory was the direct cause of AcceleratorError; keep False
    SEED        = 42

    # ── Learning rate (from-scratch recipe) ────────────────────────────────
    # Base LR scales with batch size: base_lr * (batch_size / 256)
    # base_lr = 1e-3 is the standard ViT from-scratch value
    BASE_LR      = 1e-3 * (BATCH_SIZE / 256)  # ~6.25e-5 for batch=16
    HEAD_LR      = 5e-4   # head gets a slightly higher LR
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 0.05   # strong L2, MAE-style
    WARMUP_EPOCHS = 10    # long warmup for from-scratch ViT

    # ── Regularisation ─────────────────────────────────────────────────────
    DROPOUT_HEAD1  = 0.50   # heavier dropout for small dataset
    DROPOUT_HEAD2  = 0.25
    LABEL_SMOOTH   = 0.10   # 0.10 for binary (0→0.05, 1→0.95)

    # ── MixUp & CutMix ─────────────────────────────────────────────────────
    MIXUP_ALPHA    = 0.4    # Beta(0.4, 0.4) for MixUp
    CUTMIX_ALPHA   = 1.0    # Beta(1.0, 1.0) for CutMix
    MIXUP_PROB     = 0.5    # probability of applying either augmentation
    MIXUP_CUTMIX_SPLIT = 0.5  # 50/50 split between MixUp and CutMix

    # ── Focal Loss ─────────────────────────────────────────────────────────
    FOCAL_GAMMA = 2.0   # focusing parameter: 2.0 is standard
    FOCAL_ALPHA = 0.75  # weight for positive class (ACL positive is rare)

    # ── TTA ────────────────────────────────────────────────────────────────
    TTA_N = 8           # test-time augmentation passes

    # ── Threshold ──────────────────────────────────────────────────────────
    # Optimised on validation set after training
    THRESHOLD = 0.5     # updated by find_optimal_threshold()

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CFG loaded")
print(f"  Task         : {CFG.LABEL.upper()} binary (sagittal only)")
print(f"  Pretrained   : NONE — from scratch")
print(f"  Device       : {CFG.DEVICE}")
print(f"  Base LR      : {CFG.BASE_LR:.2e}")
print(f"  Weight decay : {CFG.WEIGHT_DECAY}")
print(f"  Warmup       : {CFG.WARMUP_EPOCHS} epochs")
print(f"  Epochs       : {CFG.EPOCHS}")

CFG loaded
  Task         : ACL binary (sagittal only)
  Pretrained   : NONE — from scratch
  Device       : cuda
  Base LR      : 3.13e-05
  Weight decay : 0.05
  Warmup       : 10 epochs
  Epochs       : 120


## Section 8 — Model: ViT-Small from Scratch (~21.66M params)

Architecture: `vit_small_patch16_224` via timm.
- 12 Transformer blocks
- embed_dim = 384
- 6 attention heads, head_dim = 64
- patch_size = 16 → 14×14 = 196 patches + 1 CLS = 197 tokens
- `pretrained=False` — random initialisation, no ImageNet weights

**Weight initialisation (critical for from-scratch ViT):**
- Linear layers: truncated normal (σ=0.02), following ViT paper
- LayerNorm: weight=1, bias=0
- QKV bias: zeros
- Positional embedding: truncated normal (σ=0.02)
- CLS token: zeros

**Classification head:** LayerNorm → Linear(384, 256) → GELU → Dropout(0.5) → Linear(256, 128) → GELU → Dropout(0.25) → Linear(128, 1)
Binary output — sigmoid applied at inference, not in forward pass (BCEWithLogitsLoss handles it).

In [60]:
class ViTScratchACL(nn.Module):
    """
    ViT-Small for binary ACL classification, trained entirely from scratch.

    Architecture:
    - Backbone: timm vit_small_patch16_224 (pretrained=False)
      12 blocks, embed_dim=384, 6 heads, patch_size=16
      ~21.66M parameters
    - Head: 4-layer MLP with heavy dropout for small-dataset regularisation
      ~50K parameters
    - Total: ~21.71M parameters

    Key differences from fine-tuning setup:
    - Higher drop_path_rate (0.15 vs 0.10)
    - Heavier head dropout (0.5 vs 0.35)
    - Xavier/trunc-normal init applied to head
    - No layer freezing / progressive unfreezing needed
    """

    def __init__(self, num_classes=1):
        super().__init__()

        # Backbone — NO pretrained weights
        self.backbone = timm.create_model(
            CFG.MODEL_NAME,
            pretrained=False,          # ← from scratch
            num_classes=0,             # remove timm's default head
            drop_path_rate=CFG.DROP_PATH_RATE,
            img_size=CFG.IMAGE_SIZE,
        )

        D = self.backbone.num_features  # 384 for ViT-Small

        # Classification head — 4-layer MLP
        self.head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, 256),
            nn.GELU(),
            nn.Dropout(CFG.DROPOUT_HEAD1),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(CFG.DROPOUT_HEAD2),
            nn.Linear(128, num_classes),
        )

        # Initialise head weights explicitly
        self._init_head()

        # Apply proper initialisation to backbone (timm does this by default
        # for pretrained=True but we need to verify for pretrained=False)
        self._verify_backbone_init()

    def _init_head(self):
        """Xavier uniform for linear layers, const for norms."""
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def _verify_backbone_init(self):
        """Confirm backbone is randomly initialised (no pretrained checkpoint)."""
        # timm initialises with trunc_normal_(std=0.02) for pretrained=False
        # Patch embed weight should NOT be all zeros
        pe_weight = self.backbone.patch_embed.proj.weight
        assert pe_weight.abs().max() > 0, "Patch embed weight is all-zero — init failed"
        print(f"  Backbone init verified — patch embed std: {pe_weight.std():.4f}")

    def forward(self, x):
        """
        x : (B, 3, 224, 224) pseudo-RGB sagittal MRI
        returns: (B, 1) raw logits (apply sigmoid for probability)
        """
        features = self.backbone(x)  # (B, 384) — CLS token features
        return self.head(features)   # (B, 1)

    def get_param_groups(self, head_lr, backbone_lr, wd):
        """
        Two parameter groups:
        - Head: higher LR (randomly initialised, needs to learn faster)
        - Backbone: lower LR (random but structured init, more sensitive)

        Note: no progressive unfreezing for from-scratch. Both groups train
        from epoch 1. The backbone LR is still lower than the head to prevent
        the randomly initialised head from destabilising the backbone early.
        """
        return [
            {"params": self.head.parameters(),
             "lr": head_lr, "weight_decay": wd},
            {"params": self.backbone.parameters(),
             "lr": backbone_lr, "weight_decay": wd},
        ]

    def param_count(self):
        total     = sum(p.numel() for p in self.parameters())
        backbone  = sum(p.numel() for p in self.backbone.parameters())
        head      = sum(p.numel() for p in self.head.parameters())
        return total, backbone, head


# Instantiate
model = ViTScratchACL(num_classes=1).to(CFG.DEVICE)
total, backbone, head_p = model.param_count()

print(f"\nModel: ViT-Small from scratch (NO pretrained weights)")
print(f"  Backbone parameters : {backbone:>12,}")
print(f"  Head parameters     : {head_p:>12,}")
print(f"  Total parameters    : {total:>12,}  ({total/1e6:.2f}M)")

# Quick forward pass test
dummy = torch.randn(2, 3, 224, 224).to(CFG.DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"\nForward pass test — output shape: {tuple(out.shape)}  ✓")

  Backbone init verified — patch embed std: 0.0208

Model: ViT-Small from scratch (NO pretrained weights)
  Backbone parameters :   21,665,664
  Head parameters     :      132,353
  Total parameters    :   21,798,017  (21.80M)

Forward pass test — output shape: (2, 1)  ✓


## Section 9 — Loss, Optimiser & Scheduler

### Focal Loss for Binary Classification
Standard BCE suffers on ACL (18% positive). Focal loss adds a modulating factor that downweights easy correct predictions:
```
FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
```
- `gamma=2.0` — standard value from Lin et al. 2017 RetinaNet
- `alpha=0.75` — positive class weighted 3x more than negative

### MixUp and CutMix
Both applied stochastically (50/50 split, 50% probability per batch):
- **MixUp**: linear blend of two images and labels
- **CutMix**: paste a rectangular crop from one image into another; labels mixed proportionally by crop area

### Cosine LR with Long Warmup
From-scratch ViT requires a long linear warmup (10 epochs here) before cosine decay. Without warmup, attention weights collapse early in training.

In [61]:
class FocalBCELoss(nn.Module):
    """
    Binary Focal Loss for imbalanced ACL classification.

    Args:
        gamma (float): focusing parameter. gamma=0 → standard BCE.
                       gamma=2 reduces loss from easy examples by ~4x at p=0.9.
        alpha (float): weight for positive class. alpha=0.75 → positive
                       class contributes 3x more than negative.
        smoothing (float): label smoothing. 0.10 → targets become {0.05, 0.95}.
    """
    def __init__(self, gamma=2.0, alpha=0.75, smoothing=0.10):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.smoothing = smoothing

    def forward(self, logits, targets):
        # Label smoothing
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing

        # Standard BCE
        bce = F.binary_cross_entropy_with_logits(
            logits, targets_smooth, reduction="none"
        )

        # Compute p_t
        p    = torch.sigmoid(logits)
        p_t  = p * targets + (1 - p) * (1 - targets)

        # Alpha weighting
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        # Focal weight
        focal_weight = alpha_t * (1 - p_t) ** self.gamma

        return (focal_weight * bce).mean()


def mixup_batch(imgs, targets, alpha):
    """MixUp augmentation: linear blend of two samples."""
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(imgs.size(0))
    mixed_imgs    = lam * imgs    + (1 - lam) * imgs[perm]
    mixed_targets = lam * targets + (1 - lam) * targets[perm]
    return mixed_imgs, mixed_targets


def cutmix_batch(imgs, targets, alpha):
    """
    CutMix augmentation: replace a random rectangular region with a crop
    from another sample. Labels are mixed proportionally to area.
    """
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(imgs.size(0))
    B, C, H, W = imgs.shape

    # Random bounding box
    cut_ratio = math.sqrt(1 - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cx = random.randint(0, W)
    cy = random.randint(0, H)
    x1 = max(0, cx - cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    x2 = min(W, cx + cut_w // 2)
    y2 = min(H, cy + cut_h // 2)

    mixed = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[perm, :, y1:y2, x1:x2]

    # Adjust lambda for actual cut area
    lam = 1 - ((y2 - y1) * (x2 - x1)) / (H * W)
    mixed_targets = lam * targets + (1 - lam) * targets[perm]
    return mixed, mixed_targets


def apply_mix_augmentation(imgs, targets):
    """Apply MixUp or CutMix stochastically."""
    if random.random() >= CFG.MIXUP_PROB:
        return imgs, targets  # no augmentation
    if random.random() < CFG.MIXUP_CUTMIX_SPLIT:
        return mixup_batch(imgs, targets, CFG.MIXUP_ALPHA)
    else:
        return cutmix_batch(imgs, targets, CFG.CUTMIX_ALPHA)


def make_cosine_scheduler_with_warmup(optimizer, warmup_epochs, total_epochs, min_lr):
    """
    Linear warmup followed by cosine annealing down to min_lr.
    Critical for from-scratch ViT — without warmup attention weights collapse.
    """
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        cosine   = 0.5 * (1 + math.cos(math.pi * progress))
        # Scale cosine so it decays to min_lr, not 0
        base_lr  = optimizer.param_groups[0]["lr"]
        if base_lr > 0:
            floor = min_lr / base_lr
            return floor + (1 - floor) * cosine
        return cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Instantiate loss and optimiser
criterion = FocalBCELoss(
    gamma=CFG.FOCAL_GAMMA,
    alpha=CFG.FOCAL_ALPHA,
    smoothing=CFG.LABEL_SMOOTH
)

optimizer = torch.optim.AdamW(
    model.get_param_groups(
        head_lr=CFG.HEAD_LR,
        backbone_lr=CFG.BASE_LR,
        wd=CFG.WEIGHT_DECAY
    )
)

scheduler = make_cosine_scheduler_with_warmup(
    optimizer,
    warmup_epochs=CFG.WARMUP_EPOCHS,
    total_epochs=CFG.EPOCHS,
    min_lr=CFG.MIN_LR
)

writer = SummaryWriter(CFG.TENSORBOARD_DIR)

print("Training setup:")
print(f"  Loss      : Focal BCE (gamma={CFG.FOCAL_GAMMA}, alpha={CFG.FOCAL_ALPHA})")
print(f"  Smoothing : {CFG.LABEL_SMOOTH}")
print(f"  Optimiser : AdamW")
print(f"  Head LR   : {CFG.HEAD_LR:.2e}")
print(f"  Backbone LR: {CFG.BASE_LR:.2e}")
print(f"  Weight decay: {CFG.WEIGHT_DECAY}")
print(f"  Warmup    : {CFG.WARMUP_EPOCHS} epochs")
print(f"  Mix augment: MixUp (α={CFG.MIXUP_ALPHA}) + CutMix (α={CFG.CUTMIX_ALPHA}), p={CFG.MIXUP_PROB}")





Training setup:
  Loss      : Focal BCE (gamma=2.0, alpha=0.75)
  Smoothing : 0.1
  Optimiser : AdamW
  Head LR   : 5.00e-04
  Backbone LR: 3.13e-05
  Weight decay: 0.05
  Warmup    : 10 epochs
  Mix augment: MixUp (α=0.4) + CutMix (α=1.0), p=0.5
